# 🏡 Min-Max Normalization Workshop
## Team Name: Group 1
## Team Members: 
- Ce Chen | 9007166
- Zhuoran Zhang | xxxxxx
---

## ❗ Why We Normalize: The Problem with Raw Feature Scales

In housing data, features like `Price` and `Lot_Size` can have values in the hundreds of thousands, while others like `Num_Bedrooms` range from 1 to 5. This creates problems when we use algorithms that depend on numeric magnitudes.

---

### ⚠️ What Goes Wrong Without Normalization

---

### 1. 🧭 K-Nearest Neighbors (KNN)

KNN uses the **Euclidean distance** formula:

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + \cdots}
$$

**Example:**

- $ \text{Price}_1 = 650{,}000, \quad \text{Price}_2 = 250{,}000 $
- $ \text{Bedrooms}_1 = 3, \quad \text{Bedrooms}_2 = 2 $

Now compute squared differences:

$$
(\text{Price}_1 - \text{Price}_2)^2 = (650{,}000 - 250{,}000)^2 = (400{,}000)^2 = 1.6 \times 10^{11}
$$
$$
(\text{Bedrooms}_1 - \text{Bedrooms}_2)^2 = (3 - 2)^2 = 1
$$

➡️ **Price dominates the distance calculation**, making smaller features like `Bedrooms` irrelevant.

---

### 2. 📉 Linear Regression

Linear regression estimates:

$$
y = \beta_1 \cdot \text{Price} + \beta_2 \cdot \text{Bedrooms} + \beta_3 \cdot \text{Lot\_Size} + \epsilon
$$

If `Price` has very large values:
- Gradient updates for $ \beta_1 $ will be **much larger**
- Gradient updates for $ \beta_2 $ (Bedrooms) will be **very small**

➡️ The model overfits high-magnitude features like `Price`.

---

### 3. 🧠 Neural Networks

A single neuron computes:

$$
z = w_1 \cdot \text{Price} + w_2 \cdot \text{Bedrooms} + w_3 \cdot \text{Lot\_Size}
$$

If:

- $ \text{Price} = 650{,}000 $
- $ \text{Bedrooms} = 3 $
- $ \text{Lot\_Size} = 8{,}000 $

Then:

$$
z \approx w_1 \cdot 650{,}000 + w_2 \cdot 3 + w_3 \cdot 8{,}000
$$

➡️ Even with equal weights, `Price` contributes **most of the activation**, making it difficult for the network to learn from other features.

---

### ✅ Solution: Min-Max Normalization

We apply the transformation:

$$
x_{\text{normalized}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}
$$

This scales all features to a common range (typically $[0, 1]$).

| Feature      | Raw Value | Min     | Max     | Normalized Value |
|--------------|-----------|---------|---------|------------------|
| Price        | 650,000   | 250,000 | 800,000 | 0.72             |
| Bedrooms     | 3         | 1       | 5       | 0.50             |
| Lot_Size     | 8,000     | 3,000   | 10,000  | 0.714            |

➡️ Now, **each feature contributes fairly** to model training or distance comparisons.

---

## 📌 Use Case: Housing Data
We are normalizing features from a real estate dataset to prepare it for machine learning analysis.

In [1]:
# 🔢 Load and display dataset
import pandas as pd
df = pd.read_csv('./data/housing_data.csv')
df.head()

,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size
0,H100000,574507,1462,3,3,2002,4878
1,H100001,479260,1727,2,2,1979,4943
2,H100002,597153,1403,5,2,1952,5595
3,H100003,728454,1646,5,2,1992,9305
4,H100004,464876,853,1,1,1956,7407


### 🔎 Step 1 — Implement Min-Max Normalization on the Housing Dataset

In [2]:
# ✍️ Step 1: Implement Min-Max Normalization manually (no sklearn/numpy for scaling)
# Normalize: Price, Area_sqft, Num_Bedrooms, Num_Bathrooms, Lot_Size
# Formula: x_norm = (x - x_min) / (x_max - x_min)

cols_to_normalize = ['Price', 'Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']

df_norm = df.copy()

for col in cols_to_normalize:
    x_min = df[col].min()
    x_max = df[col].max()
    denom = (x_max - x_min)
    if denom == 0:
        # Edge case: constant column
        df_norm[col + '_norm'] = 0.0
    else:
        df_norm[col + '_norm'] = (df[col] - x_min) / denom

# Quick checks: each normalized column should be within [0, 1]
print("Normalized column ranges (min, max):")
for col in cols_to_normalize:
    c = col + '_norm'
    print(f"{c:16s} -> ({df_norm[c].min():.4f}, {df_norm[c].max():.4f})")

display(df_norm[['House_ID'] + cols_to_normalize + [c + '_norm' for c in cols_to_normalize]].head())

# Optional: save normalized data for later steps
df_norm.to_csv('./data/housing_data_with_minmax.csv', index=False)
print("Saved: ./data/housing_data_with_minmax.csv")


Normalized column ranges (min, max):
Price_norm       -> (0.0000, 1.0000)
Area_sqft_norm   -> (0.0000, 1.0000)
Num_Bedrooms_norm -> (0.0000, 1.0000)
Num_Bathrooms_norm -> (0.0000, 1.0000)
Lot_Size_norm    -> (0.0000, 1.0000)


,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Lot_Size,Price_norm,Area_sqft_norm,Num_Bedrooms_norm,Num_Bathrooms_norm,Lot_Size_norm
0,H100000,574507,1462,3,3,4878,0.485226,0.315789,0.50,1.0,0.320814
1,H100001,479260,1727,2,2,4943,0.387827,0.394588,0.25,0.5,0.326191
2,H100002,597153,1403,5,2,5595,0.508384,0.298246,1.00,0.5,0.380129
3,H100003,728454,1646,5,2,9305,0.642651,0.370503,1.00,0.5,0.687045
4,H100004,464876,853,1,1,7407,0.373119,0.134701,0.00,0.0,0.530030


Saved: ./data/housing_data_with_minmax.csv


### 🔎 Talking Points #1

- After Min-Max normalization, each selected feature is rescaled into the same range **[0, 1]**, so large‑magnitude features (e.g., `Price`) do not dominate distance‑based or gradient‑based models just because of their units.
- Min-Max normalization preserves the **shape of the distribution** (it is a linear transformation), but it can be **sensitive to outliers** because extreme min/max values compress the majority of the data into a narrower interval.
- A quick validation step is to print **(min, max)** for each normalized column and confirm they are approximately **(0, 1)**, which helps catch mistakes (e.g., dividing by the wrong denominator or normalizing the wrong columns).



## 🧩 Challenge Extension: After Normalization, Which Features Matter Most?

You’ve normalized the housing features so they share a common scale.  
Now comes a common next step in ML workflows:

> **How do we identify the most important directions (principal components) in the data—and how might these relate to a target variable like `Price`?**

This introduces **Principal Component Analysis (PCA)**.

---

## 📚 PCA Theory (Conceptual)

### What PCA *is*
PCA is an **unsupervised** dimensionality reduction technique that:
- Finds **new axes** (principal components) that are **linear combinations** of your original features.
- Orders these axes so that:
  - **PC1** captures the **most variance** in the feature space,
  - **PC2** captures the next most variance, and so on,
  - Each PC is **orthogonal** (uncorrelated) with the previous ones.

### What PCA is *not*
PCA does **not** directly find features that “impact the target variable” because it does not use the target in its optimization.

However, you *can*:
- Compute PCs from the feature matrix **X** (after normalization),
- Then measure how PCs relate to the target **y** (e.g., correlation with `Price`, or a simple regression on PCs),
- Interpret which original features contribute most to PCs that are most related to **y**.

---

## 🧠 The Math (high level)
Given a centered feature matrix \(X\) (often standardized/normalized first):

1. Compute covariance matrix:
$$
\Sigma = \frac{1}{n-1}X^\top X
$$

2. Find eigenvectors (principal directions) and eigenvalues (variance captured):
$$
\Sigma v_i = \lambda_i v_i
$$

- $ v_i $ are **principal component directions** (loadings)
- $ \lambda_i $ are the **variance explained** by each component

---

## ✅ Why Normalize Before PCA?
PCA is sensitive to scale. Without normalization/standardization:
- A large-scale feature (e.g., `Price`) can dominate variance
- PCs will reflect units rather than structure

---

## 🎯 Student Challenge
Using the **housing dataset**:

1. Apply PCA to the normalized feature matrix \(X\) (exclude ID columns and the target).
2. Determine how many components are needed to explain **≥ 90%** of the variance.
3. Identify which original features contribute most to:
   - **PC1** and **PC2**, and
   - the **PC most correlated with the target** (`Price`).
4. Write a short interpretation:
   - “What does PC1 represent in housing terms?”
   - “Do the PCs that explain the most variance also relate most strongly to `Price`?”



### 🔗 How to Integrate This With Your Step 1 Normalization

- If you created normalized columns (e.g., `Area_sqft_norm`), use those in `candidate_features`.
- If you normalized in-place (overwriting original columns), you can use the original names.
- PCA should **not** include:
  - `House_ID` (identifier)
  - non-numeric categorical columns (unless encoded appropriately)
- Decide intentionally whether to include `Year_Built`:
  - It’s numeric, but it may behave differently than size/price-related features.

---

### ✅ Deliverable for the Challenge
Add a Markdown cell answering:

1. How many PCs explain at least **90%** variance?
2. Which features contribute most to **PC1** and **PC2**?
3. Which PC is most correlated with `Price`?
4. In plain language: what do you think PC1 represents?


In [3]:
# --- Student Challenge: After Normalization, Which Features Matter Most? ---
# Goal: Use PCA to see which normalized features explain the most variation,
# and check whether principal components relate to the target (Price).

import pandas as pd
import numpy as np

from sklearn.decomposition import PCA

# ---- Step A: Choose target and feature columns ----
target_col = 'Price'  # keep raw Price for interpretability
feature_cols_norm = ['Area_sqft_norm', 'Num_Bedrooms_norm', 'Num_Bathrooms_norm', 'Lot_Size_norm']

X = df_norm[feature_cols_norm].values
y = df_norm[target_col].values  # raw price

# ---- Step B: Scaling for PCA ----
# Since Step 1 already puts features in [0, 1], they are on a comparable scale.
# (If features were not normalized, you would typically standardize them before PCA.)

# ---- Step C: Fit PCA ----
pca = PCA(n_components=len(feature_cols_norm), random_state=0)
X_pca = pca.fit_transform(X)

# ---- Step D: Variance explained ----
explained = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(feature_cols_norm))],
    'Explained Variance Ratio': pca.explained_variance_ratio_,
    'Explained Variance': pca.explained_variance_,
})
display(explained)

print(f"Total explained variance ratio (sum): {pca.explained_variance_ratio_.sum():.4f}")

# ---- Step E: Loadings (feature contributions to PCs) ----
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols_norm,
    columns=[f'PC{i+1}' for i in range(len(feature_cols_norm))]
)

# Show absolute loadings to identify the strongest contributors
abs_loadings = loadings.abs().sort_values(by='PC1', ascending=False)
display(loadings)
display(abs_loadings)

# For each PC, show the top contributing feature
top_features_per_pc = abs_loadings.apply(lambda col: col.idxmax())
print("Top contributing feature per PC:")
for pc, feat in top_features_per_pc.items():
    print(f"  {pc}: {feat}")

# ---- Step F: Relate PCs to the target (simple correlation with Price) ----
pc_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(len(feature_cols_norm))])
corr_with_price = pc_df.apply(lambda s: s.corr(pd.Series(y)))
display(pd.DataFrame({'Corr(PC, Price)': corr_with_price}))

print("Feature correlation matrix (normalized features):")
display(df_norm[feature_cols_norm].corr())


,PC,Explained Variance Ratio,Explained Variance
0,PC1,0.495319,0.170683
1,PC2,0.360045,0.124069
2,PC3,0.080260,0.027657
3,PC4,0.064376,0.022184


Total explained variance ratio (sum): 1.0000


,PC1,PC2,PC3,PC4
Area_sqft_norm,0.001548,0.017423,-0.123563,0.992183
Num_Bedrooms_norm,0.053396,0.998253,0.020288,-0.015087
Num_Bathrooms_norm,0.998563,-0.053326,-0.005191,-0.001268
Lot_Size_norm,0.004326,-0.018523,0.992116,0.123874


,PC1,PC2,PC3,PC4
Num_Bathrooms_norm,0.998563,0.053326,0.005191,0.001268
Num_Bedrooms_norm,0.053396,0.998253,0.020288,0.015087
Lot_Size_norm,0.004326,0.018523,0.992116,0.123874
Area_sqft_norm,0.001548,0.017423,0.123563,0.992183


Top contributing feature per PC:
  PC1: Num_Bathrooms_norm
  PC2: Num_Bedrooms_norm
  PC3: Lot_Size_norm
  PC4: Area_sqft_norm


,"Corr(PC, Price)"
PC1,-0.011395
PC2,-0.005423
PC3,0.017210
PC4,-0.015174


Feature correlation matrix (normalized features):


,Area_sqft_norm,Num_Bedrooms_norm,Num_Bathrooms_norm,Lot_Size_norm
Area_sqft_norm,1.000000,0.033654,0.002243,-0.028328
Num_Bedrooms_norm,0.033654,1.000000,0.017138,-0.029712
Num_Bathrooms_norm,0.002243,0.017138,1.000000,0.010403
Lot_Size_norm,-0.028328,-0.029712,0.010403,1.000000


### 🔎 Talking Points #2

- PCA on the normalized feature set shows that the first two principal components explain about **85.5%** of the variance (PC1 ≈ **49.5%**, PC2 ≈ **36.0%**), so most variation can be summarized with a small number of components.
- The loading matrix indicates that each principal component is dominated by a single feature (because the normalized features have very low pairwise correlation), meaning PCA is mostly **re‑expressing** the original variables rather than discovering a strong combined “size” factor.
- The correlations between PCs and `Price` are close to zero in this dataset, suggesting that these specific features (as generated here) do not show a strong linear relationship with `Price`, so PCA does not reveal a clear “most important” driver for predicting price.
